# Geheugentransfers bij precisiewissels
## Zes bestaande SVHN CNN-8-modellen · initiële loading, gewichten en activaties

Deze notebook presenteert **berekend geheugenverkeer** uit `run_check.py run`.
Ze start geen training, netwerk-inference of nieuwe accuracy-evaluatie.

**Hoofdvraag:** wat verandert aan opslag en parametertransfers wanneer verschillende
precisies uit één truncatiebron worden afgeleid, bij dezelfde beschikbare bufferruimte?

De figuren scheiden de **initiële laadfase** van het vervolg, en houden
**parameterverkeer en activatieverkeer apart**. De resultaten beschrijven expliciete
voorgestelde opslagindelingen en een geheugenmodel; het zijn geen GPU- of hardwaremetingen.


## 1. Resultaten openen

Open deze notebook bij voorkeur in de nieuwe resultatenmap en kies **Run All**.
Daar staan ook `run_results.json` en `analysis_helpers.py`.
`RESULT_DIR = None` gebruikt eerst die map, anders de recentste geslaagde proef onder
`mx_runs/mx_precision_switch_transfer_eval`. Vul voor een specifieke run het pad in.

De runner gebruikt alleen standaard-Python; deze notebook gebruikt NumPy, pandas en Matplotlib.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

RESULT_DIR = None  # Bijvoorbeeld Path(r"C:\Users\renzo\master-thesis-mx\mx_runs\mx_precision_switch_transfer_eval\..._run")
SAVE_FIGURES = True

# Normaal staat de helper naast de notebook; ook openen vanuit de projectmap is mogelijk.
for candidate in [Path.cwd(), Path.cwd()/"software_checks"/"mx_precision_switch_transfer_eval_v1"]:
    if (candidate/"analysis_helpers.py").is_file():
        sys.path.insert(0, str(candidate))
        break
import analysis_helpers as h
h.style()
RESULT_ROOT, results, tables = h.load_results(RESULT_DIR)
FIG_DIR = RESULT_ROOT/"figures" if SAVE_FIGURES else None
LINE_BYTES = results["config"]["main_line_bytes"]
CACHE_KIB = 128 if 128 in results["config"]["cache_kib"] else results["config"]["cache_kib"][-1]
print("Resultaten:", RESULT_ROOT)
print("Berekende combinaties:", len(tables["summary"]))
print("Simulatietijd [s]:", round(results["simulation_seconds"], 3))


In [ ]:
status = results.get("status", "UNKNOWN")
source_status = results.get("source_validation", {}).get("status", "NOT_CHECKED")
if status == "PASS" and source_status == "PASS":
    display(Markdown("**Broncontrole: PASS.** De zes bestaande trainingen/checkpointbestanden zijn bevestigd. De verkeerscijfers blijven modelberekeningen; er zijn geen checkpointtensors naar een nieuwe fysieke export geconverteerd."))
else:
    display(Markdown(f"**REFERENTIEBEREKENING — lokale broncontrole niet bevestigd ({status}; {source_status}).** Gebruik deze uitvoer niet als een brongeverifieerde run op de eigen checkpoints."))


## 2. Modellen en afspraken

- **GitHub:** vijf gekoppelde W/A-modi (32, 8, 4, 2, 1).
- **Twee MX-modi:** E2M3/E2M1 rechtstreeks, of via mantissatruncatie; B32.
- **Drie MX-modi:** E4M3/E2M3/E2M1 rechtstreeks bij B32, of via exponent- en
  mantissatruncatie met vaste ×64-schaal bij B32 en B64.
- Alle MX-modi behouden **A32**. Eerste en laatste laag blijven FP32.

De hoofdvergelijking gebruikt **64 verschillende invoerbeelden**, batchgrootte 1,
en afwisselend de hoogste en laagste modus. De absolute precisies verschillen tussen
de families. Vergelijk het truncatie-effect daarom vooral binnen de twee MX-modi en
binnen de drie B32-MX-modi. B64 verandert bovendien de groepsgrootte en padding.

De parametercache is volledig associatief met LRU-vervanging. Iedere laag vraagt
haar parameterarrays eenmaal op; hergebruik binnen de laag wordt aangenomen.
Activatiebuffers zijn afzonderlijk en verdringen geen parameterlijnen.


In [ ]:
display(pd.DataFrame(results["models"])[["label", "modes", "group_size", "activation_policy", "route"]])
display(pd.DataFrame({"Instelling": ["Inferences", "Batchgrootte", "Hoofdtransfer [bytes]", "Parametercache [KiB]", "Hoofdfiguren [KiB]"],
                      "Waarde": [results["config"]["n_inferences"], results["config"]["batch_size"], LINE_BYTES,
                                 str(results["config"]["cache_kib"]), CACHE_KIB]}))


## 3. Compacte operationele opslag

De directe modellen bewaren compacte codes per modus en lezen alleen de gewenste
modus. Gelijke schaalarrays van direct E2M3/E2M1 worden gedeeld; eerste/laatste FP32-lagen
zijn eveneens gedeeld. Iedere modus heeft haar eigen actieve BN-bank.

Truncatiemodellen bewaren één bron. De hoofdvariant groepeert die bronbits in
afzonderlijk leesbare codedelen. Een controlevariant vraagt steeds de volledige bron op.

Hier tellen uitsluitend benodigde inferenceparameters mee. De eerdere meetingnotebook
telde alle vijf BN-banken en BN-tellers in de exportkosten; daarom kunnen de totalen
klein verschillen. Trainingsmasters, Adam, bestandsheaders en containers tellen hier niet mee.


In [ ]:
h.render(h.storage_plot(tables), "01_operationele_opslag", FIG_DIR)


## 4. Initiële loading — afzonderlijk zichtbaar

Deze figuur vergelijkt een koude lezing van **alleen de hoogste modus**, **alleen de
laagste modus**, of **alle operationele modidata**. Uitlijning op hele transferlijnen
wordt meegerekend. Alle modi vooraf lezen is een beleidskeuze; het is niet vereist
voor één inference op één precisie.


In [ ]:
h.render(h.loading_plot(tables), "02_initiele_loading", FIG_DIR)


## 5. Hoofdvergelijking — wisselen tussen hoogste en laagste precisie

**Op aanvraag:** de cache start leeg; de eerste inference vormt de initiële laadfase,
de overige inferences vormen het vervolg.

**Alles vooraf lezen:** eerst alle modidata door dezelfde cache lezen, daarna de
volledige reeks uitvoeren. Als alles niet past, verdringt voorladen ook gegevens.
Deze vooraf gelezen bytes worden volledig meegeteld.

In beide gevallen geldt **initiële laadfase + vervolg = totaal**. De figuur toont
parametertransfers inclusief gewichtscodes, schalen, gedeelde FP32-lagen en actieve BN.


In [ ]:
h.render(h.switching_plot(tables, CACHE_KIB, LINE_BYTES), "03_precisiewissels_initieel_en_vervolg", FIG_DIR)


## 6. Hoe belangrijk is de beschikbare buffer?

Een compactere bron kan in een kleinere cache passen en bij precisiewissels worden
hergebruikt. Een lage precisie levert echter niet automatisch minder transfers op:
bij weinig cache, uitlijning of het ophalen van de volledige bron kan dat voordeel verdwijnen.

De x-as toont de afzonderlijk onderzochte capaciteiten. De gemiddelde transfers
bevatten ook de koude start; de hitratio telt aangevraagde parameterlijnen.


In [ ]:
h.render(h.cache_plot(tables, LINE_BYTES), "04_cachegevoeligheid", FIG_DIR)


## 7. Het verloop over de volledige wisselreeks

De gemarkeerde eerste inference is de initiële laadfase. Een vlakker vervolg betekent
meer hergebruik van reeds aanwezige parameterlijnen. De drie panelen hebben een eigen
verticale schaal; lees de labels bij een vergelijking tussen families.


In [ ]:
h.render(h.cumulative_plot(tables, CACHE_KIB, LINE_BYTES), "05_cumulatieve_precisiewissels", FIG_DIR)


## 8. Controle — selectieve codedelen of de volledige bron ophalen

Dit zijn twee uitleesvarianten van **hetzelfde truncatiemodel**, met dezelfde codes en schalen.
Bij de selectieve variant vraagt een lagere modus alleen de benodigde delen op.
Bij de volledige-bronvariant wordt de hele E2M3- of E4M3-bron aangevraagd; de cache kan
die bron vervolgens hergebruiken.

De universele bitmapping wordt in de zelftests gecontroleerd. Deze proef maakt geen
nieuwe checkpoint-export en meet geen nieuwe accuracy.


In [ ]:
h.render(h.full_source_plot(tables, LINE_BYTES), "06_selectief_of_volledige_bron", FIG_DIR)


## 9. Activatieverkeer — apart van de gewichten

De daadwerkelijke netwerkinput is **3×40×40** door de bestaande `Resize(40)`-transformatie.
Het model leest per samengevoegde conv/BN/pool/activatiefase de ingang eenmaal en schrijft
de uitgang eenmaal. Interne tussenresultaten, im2col, tile-herlezingen en spillverkeer
vallen buiten dit eenvoudige model.

GitHub varieert ook A; onze MX-modellen blijven A32. Voor lage GitHub-modi nemen we
compacte activatieopslag aan. De huidige PyTorch-code gebruikt daar ook FP32-tensors.
Invoer, de laatste `Activate(quantize=False)` en logits blijven FP32.
Activaties worden niet tussen verschillende invoerbeelden hergebruikt.


In [ ]:
h.render(h.activation_plot(tables, LINE_BYTES), "07_activaties_afzonderlijk", FIG_DIR)


## 10. Gevoeligheid voor transfergrootte

Een geheugensysteem haalt hele woorden/lijnen op. Minder nuttige bits betekent daarom
niet altijd evenredig minder verkeer. Deze controle houdt de parametercache en
precisiereeks gelijk en verandert alleen de transfergrootte.


In [ ]:
h.render(h.line_plot(tables, results["config"]["sensitivity_cache_kib"]), "08_transfergrootte", FIG_DIR)


## 11. Aanvullende precisiereeksen

`alternating_extremes`: iedere inference wisselen tussen hoog en laag.
`dwell_extremes`: meerdere opeenvolgende inferences op dezelfde precisie.
`cycle_all`: alle ondersteunde modi cyclisch gebruiken.

De laatste reeks bevat verschillende precisiemengsels bij twee, drie en vijf modi.
Gebruik ze als aanvullende gebruikssituatie, niet als identiek werkaanbod.
Gewicht- en activatieverkeer blijven afzonderlijke kolommen.


In [ ]:
display(h.scenario_table(tables, CACHE_KIB, LINE_BYTES).round(3))


## 12. Accuracycontext van de bestaande epoch-100-checkpoints

Onderstaande accuracies komen uit de eerder voltooide trainingen. Ze worden hier
uitsluitend als context overgenomen; de geheugenproef voert geen SVHN-evaluatie uit.
Het geheugenmodel hangt bij deze vaste dichte indelingen af van vormen en formaten,
niet van de individuele gewichtswaarden.


In [ ]:
accuracy=tables["accuracy"]
if len(accuracy):
    display(accuracy[["model", "format", "top1_percent", "correct1", "n", "run_id"]].rename(
        columns={"top1_percent":"Epoch-100-testaccuracy [%]"}).round(4))
else:
    print("Geen accuracy_context.json bij deze resultaatmap gevonden; geen waarden ingevuld.")


## 13. Wat deze proef wel en niet vaststelt

De resultaten geven opslag, nuttige payloadbytes, cachehergebruik en transfers onder
de vermelde aannames. De broncontrole koppelt de proef aan bestaande afgeronde runs.
De bitmappingcontrole voorkomt dat de voorgestelde codedelen een andere truncatieroute voorstellen.

Nog niet gemeten zijn werkelijke hardwarecycli, energie, decoderkosten, bankconflicten,
intralayer tile-herlezingen en training met FP32-masters/Adam. Ook een gedeelde
parameter-/activatiecache kan andere resultaten opleveren dan de afzonderlijke buffers hier.

De uiteindelijke conclusie moet daarom aangeven **bij welke cachecapaciteit,
precisiereeks, laadkeuze en transfergrootte** truncatie verkeer vermindert, gelijk houdt
of verhoogt. Een lagere opslag alleen is nog geen bewijs voor lager uitvoeringsverkeer.


In [ ]:
# Volledige controleerbare tabellen blijven beschikbaar; niets wordt opnieuw gesimuleerd.
display(tables["summary"][["model_label", "variant", "scenario", "line_bytes", "cache_kib", "preload_policy",
                           "initial_load_bytes", "continuation_bytes", "weight_total_bytes",
                           "activation_total_bytes"]].head(18))
assert (tables["summary"].initial_load_bytes + tables["summary"].continuation_bytes == tables["summary"].weight_total_bytes).all()
print("Controle: initiële loading + vervolg = totaal voor alle combinaties.")
if FIG_DIR is not None: print("PNG/SVG-figuren:", FIG_DIR)
